# zyc_cn_mandarin — Kaggle 训练

和本地 Mac 那套**完全相同的配方**，唯一差别是跑在 CUDA 上。
目的是判断本地的电音问题是不是 MPS 造成的。

## 开始之前

1. 右侧 **Session options → Accelerator** 选 **GPU T4 x2** 或 **GPU P100**
2. 右侧 **Input → Add Input → Upload** 把 `zyc_dataset_for_cloud.zip` 传上来
3. 从上到下依次运行

预计 200 epoch 约 1–2 小时。

## 1. 确认 GPU

In [ ]:
!nvidia-smi
import torch, sys
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "没有 GPU —— 去 Session options 里把 Accelerator 设成 GPU"


## 2. 安装 Applio

约 5–8 分钟。

In [ ]:
%cd /kaggle/working
!rm -rf Applio
!git clone --depth 1 https://github.com/IAHispano/Applio.git
%cd /kaggle/working/Applio

# 关键：绝对不要动 Kaggle 镜像自带的 torch —— 它是和本机 CUDA 配好的。
# Applio 的 requirements.txt 里 pin 了 torch==2.11.0 / torchaudio==2.11.0（PyPI 版，
# 不带 CUDA），装上会覆盖 GPU 版，然后 torchaudio 加载时报
#   OSError: libcudart.so.XX: cannot open shared object file
# 所以这里把 torch 系列从 requirements 里剔除再装。
!grep -vE '^(torch|torchaudio|torchvision)==' requirements.txt > /kaggle/working/req_notorch.txt
!echo "--- 剔除的行 ---" && grep -E '^(torch|torchaudio|torchvision)==' requirements.txt
!pip install -q -r /kaggle/working/req_notorch.txt

import torch, torchaudio, torchvision
print("\ntorch      ", torch.__version__, "| cuda 可用:", torch.cuda.is_available())
print("torchaudio ", torchaudio.__version__)
assert torch.cuda.is_available(), "torch 被覆盖了 —— 需要 Factory reset 重来"
print("✅ GPU 版 PyTorch 完好")


## 3. 下载前置模型

音高预测器 rmvpe.pt + HiFi-GAN 预训练底模。
**这一步不能跳** —— 本地就是因为漏了它，特征提取静默失败过。

In [ ]:
%cd /kaggle/working/Applio
!python core.py prerequisites --models --pretraineds-hifigan --no-exe

# assets/config.json 只有开 GUI 才会生成，命令行不会。缺了它导出模型那步会静默失败。
import shutil, os
cfg = "assets/config.json"
if not os.path.exists(cfg):
    shutil.copy("assets/config_template.json", cfg)
    print("已从模板生成 assets/config.json")


## 4. 找到数据集并解压

In [ ]:
import glob, os, zipfile, shutil

DATASET_DIR = "/kaggle/working/dataset"
shutil.rmtree(DATASET_DIR, ignore_errors=True)
os.makedirs(DATASET_DIR, exist_ok=True)
EXTS = (".wav", ".flac", ".mp3", ".ogg", ".m4a")

# 1) 解压 input 下所有 zip
for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
    print("解压", os.path.basename(z))
    with zipfile.ZipFile(z) as f:
        f.extractall("/kaggle/working/_unzipped")

# 2) 从解压结果 + input 目录本身收集音频，摊平到 DATASET_DIR
seen = set(); n = 0
for root in ["/kaggle/working/_unzipped", "/kaggle/input"]:
    for dirpath, _, files in os.walk(root):
        for f in files:
            if not f.lower().endswith(EXTS) or f.startswith("."):
                continue
            if f in seen:            # 同名只取一次，避免 zip 内外重复
                continue
            seen.add(f)
            shutil.copy(os.path.join(dirpath, f), os.path.join(DATASET_DIR, f))
            n += 1

print(f"\n数据集: {DATASET_DIR}  共 {n} 个音频文件")
for f in sorted(os.listdir(DATASET_DIR))[:5]:
    print("  ", f)
print("   ...")
assert n >= 20, f"只找到 {n} 个文件，预期 24 个 —— 检查上传是否完整"


## 5. 参数

**和本地完全一致，别改** —— 改了就失去对照意义。

只有 `--gpu 0` 和本地不同：本地 Mac 必须传 `-`（Applio 的 extract.py 会把非 `-` 的值拼成 `cuda:N`），
这里是真 CUDA，所以传 `0`。

In [ ]:
MODEL_NAME   = "zyc_cn_mandarin_dry"
SAMPLE_RATE  = 40000
BATCH_SIZE   = 8
TOTAL_EPOCH  = 200
SAVE_EVERY   = 25
F0_METHOD    = "rmvpe"
EMBEDDER     = "chinese-hubert-base"
VOCODER      = "HiFi-GAN"
NORMALIZATION= "pre"           # 切片器 -42dB 门槛会裁掉轻的段落，pre 归一后切片数才够
GPU          = "0"             # 本地 Mac 是 "-"
print("配置就绪")


## 6. 预处理（切片 + 重采样）

约 1 分钟。

In [ ]:
%cd /kaggle/working/Applio
!python core.py preprocess \
  --model-name "{MODEL_NAME}" \
  --dataset-path "/kaggle/working/dataset" \
  --sample-rate {SAMPLE_RATE} \
  --cut-preprocess "Automatic" \
  --chunk-len 3.0 --overlap-len 0.3 \
  --normalization-mode "{NORMALIZATION}"

import os, glob
sl = glob.glob(f"logs/{MODEL_NAME}/sliced_audios_16k/*.wav")
print(f"切片数: {len(sl)}")
assert len(sl) > 0, "预处理失败 —— Applio 失败时退出码仍是 0，必须查产物"


## 7. 特征提取

约 2–5 分钟。

In [ ]:
!python core.py extract \
  --model-name "{MODEL_NAME}" \
  --sample-rate {SAMPLE_RATE} \
  --f0-method "{F0_METHOD}" \
  --embedder-model "{EMBEDDER}" \
  --gpu "{GPU}" \
  --include-mutes 2

import os
fl = f"logs/{MODEL_NAME}/filelist.txt"
n = sum(1 for _ in open(fl))
print(f"filelist: {n} 行  ->  每 epoch {n//BATCH_SIZE} 步")
assert n > 0, "filelist 是空的 —— 音高提取失败了（多半是 rmvpe.pt 没下载）"


## 8. 生成索引

In [ ]:
!python core.py index --model-name "{MODEL_NAME}" --index-algorithm "Auto"
import glob
print(glob.glob(f"logs/{MODEL_NAME}/*.index"))


## 9. 训练

200 epoch，每 25 存一次档。`--save-every-weights` 默认开启，所以每个存档点都会导出可直接用的 `.pth`。

**中途可以停** —— 想早点试听就中断这个单元格，然后直接跑第 10 步打包，已有的存档都能用。

In [ ]:
!python core.py train \
  --model-name "{MODEL_NAME}" \
  --sample-rate {SAMPLE_RATE} \
  --batch-size {BATCH_SIZE} \
  --total-epoch {TOTAL_EPOCH} \
  --save-every-epoch {SAVE_EVERY} \
  --gpu "{GPU}" \
  --vocoder "{VOCODER}" \
  --pretrained \
  --index-algorithm "Auto"


## 10. 打包下载

跑完后在右侧 **Output** 里下载 zip。

In [ ]:
import glob, os, zipfile
OUT = "/kaggle/working/zyc_trained.zip"
files = sorted(glob.glob(f"logs/{MODEL_NAME}/{MODEL_NAME}_*e_*s.pth")) \
      + glob.glob(f"logs/{MODEL_NAME}/*.index")
print("要打包的文件:")
for f in files:
    print(f"  {os.path.basename(f)}  {os.path.getsize(f)/1048576:.1f} MB")
assert files, "没有产物 —— 检查上面的训练日志"
with zipfile.ZipFile(OUT, "w", zipfile.ZIP_STORED) as z:
    for f in files:
        z.write(f, os.path.basename(f))
print(f"\n打包完成: {OUT}  {os.path.getsize(OUT)/1048576:.0f} MB")


## 拿回本地后

把解压出来的文件放到：

```
~/workspace/Applio/logs/zyc_cn_mandarin_cloud/
```

然后本地推理：

```bash
uv run python -c "
import sys; sys.path.insert(0,'mac')
import settings, cover
A = settings.load()['applio_dir']
pth, idx = cover.find_trained_model('zyc_cn_mandarin_cloud', A, 'models')
cover.rvc_infer('output/_同域测试/输入_我的原声.flac', 'output/_同域测试/输出_云端.flac',
                pth, idx, embedder_model='chinese-hubert-base', reverb=False, applio_dir=A)
"
```

**判断标准：** 拿它和本地的 `同域_1e-4_旧.flac` 对比。

- 云端版没电音 → 是 MPS 的问题，以后训练都放云端，本地只做分离和推理
- 云端版一样有电音 → 是数据量的问题，需要把素材补到 60 分钟以上